In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.signal import butter, lfilter, welch
from scipy.stats import entropy
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

# --- EEG band definitions ---
bands = {
    'alpha': (8, 13),  # We focus only on the alpha band
}

# --- Bandpass filter function ---
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data)

# --- Feature extraction function ---
def extract_alpha_features(signal, fs):
    features = {}
    for band, (low, high) in bands.items():
        filtered = bandpass_filter(signal, low, high, fs)
        psd_freqs, psd = welch(filtered, fs)

        # Calculate histogram-based entropy
        hist, _ = np.histogram(filtered, bins=50, density=True)
        hist = hist[hist > 0]
        band_entropy = -np.sum(hist * np.log2(hist))

        features[band] = {
            'mean': np.mean(filtered),
            'variance': np.var(filtered),
            'entropy': band_entropy,
            'psd': np.mean(psd)
        }
    return features

# --- Load OpenBCI .easy file robustly ---
def load_easy_file(filepath):
    with open(filepath, 'r') as file:
        lines = file.readlines()

    # Count header rows
    skip_rows = 0
    for line in lines:
        if line.startswith('%') or line.startswith('#'):
            skip_rows += 1
        else:
            break

    # Try common separators
    for sep in [',', '\t', ';']:
        try:
            df = pd.read_csv(filepath, skiprows=skip_rows, sep=sep)
            if df.shape[1] > 1:
                print(f"Loaded using separator: '{sep}'")
                return df
        except Exception as e:
            print(f"Failed with separator '{sep}': {e}")

    raise ValueError("Failed to read the file with common separators.")

# --- Process dataset and save to CSV ---
def process_and_save_alpha_features(dataset_path, output_csv, fs=256):
    # List to store all feature rows
    all_features = []

    # Iterate over each file in the dataset
    for label in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label)
        if not os.path.isdir(label_path):
            continue

        for file in os.listdir(label_path):
            if file.endswith('.easy'):
                filepath = os.path.join(label_path, file)

                try:
                    # Load EEG data from file
                    df = load_easy_file(filepath)

                    # Choose an EEG channel (this can be adjusted based on your data)
                    eeg_channel = df.columns[1]  # Modify if needed
                    eeg_data = df[eeg_channel].values

                    # Extract alpha band features for the signal
                    features = extract_alpha_features(eeg_data, fs)

                    # Prepare the row for the current file
                    feature_row = {'filename': file, 'label': label}
                    for key, val in features['alpha'].items():
                        feature_row[f"alpha_{key}"] = val

                    # Add this row to the list of features
                    all_features.append(feature_row)

                except Exception as e:
                    print(f"Error processing file {file}: {e}")

    # Convert feature list to DataFrame and save to CSV
    df_features = pd.DataFrame(all_features)
    df_features.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

# --- SVM Classification ---
def apply_svm_classifier(input_csv):
    # Load the extracted features from the CSV
    df = pd.read_csv(input_csv)
    print(f"Data loaded from {input_csv}")
    print(df.head())

    # Prepare the features (X) and labels (y)
    X = df[['alpha_mean', 'alpha_variance', 'alpha_entropy', 'alpha_psd']].values
    y = df['label'].values

    # Encode labels (e.g., 'mediator' -> 0, 'non_mediator' -> 1)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

    # Apply PCA to reduce features to 2D for visualization
    pca = PCA(n_components=2)
    X_train_2d = pca.fit_transform(X_train)
    X_test_2d = pca.transform(X_test)

    # --- Initialize and train SVM classifier ---
    clf = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf.fit(X_train, y_train)

    # Make predictions
    y_pred = clf.predict(X_test)

    # --- Create meshgrid for decision boundary plot ---
    h = 0.1
    x_min, x_max = np.clip(X_train_2d[:, 0].min() - 1, -10, 10), np.clip(X_train_2d[:, 0].max() + 1, -10, 10)
    y_min, y_max = np.clip(X_train_2d[:, 1].min() - 1, -10, 10), np.clip(X_train_2d[:, 1].max() + 1, -10, 10)
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Predict using the 2D SVM
    clf_2d = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf_2d.fit(X_train_2d, y_train)
    Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot decision boundary
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)
    plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap=plt.cm.coolwarm, edgecolors='k')
    plt.title("SVM Decision Boundary (2D PCA Projection)")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.grid(True)
    plt.show()

    # Plot the support vectors
    plt.scatter(clf_2d.support_vectors_[:, 0], clf_2d.support_vectors_[:, 1], s=100,
                linewidth=1, facecolors='none', edgecolors='k', label='Support Vectors')

    # Labels and title
    plt.title("SVM Decision Boundary with PCA-reduced Features")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.legend(["Train Data", "Test Data", "Support Vectors"], loc="upper left")
    plt.grid(True)
    plt.show()

    # Classification report
    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    # Confusion Matrix
    ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, display_labels=le.classes_)
    plt.title("Confusion Matrix")
    plt.show()

# --- Main Execution ---
dataset_path = "/kaggle/input/datasetik/Data"  # Your dataset path
output_csv = "alpha_band_features.csv"  # Output CSV filename
fs = 256  # Set your actual sampling frequency

# Step 1: Process the dataset and save features to CSV
process_and_save_alpha_features(dataset_path, output_csv, fs)

# Step 2: Apply SVM Classifier on the extracted features
apply_svm_classifier(output_csv)
